# Best Model Analysis — CrossLead Deeper, filters=(48, 96, 192)

Loads the most performant model across all 3-stage Optuna runs:

**Source:** `optuna_crosslead_3stage_filters/2026-04-27_10-29-59/best_model.pt`

**Config:**
- `stage_filters=(48, 96, 192)` — 965K params (filter study winner)
- `kernels=(7, 5, 3)` — RF=180 ms
- `lr=2.465e-3, dropout=0.0546, weight_decay=1.67e-4`
- `aug_sigma=0.060, aug_max_shift=276` (training)
- `n_heads=4`

**Reported metrics:** CV AUROC 0.7034, Test AUROC **0.7952**

Sections:
1. Train vs test distribution
2. ROC + Precision-Recall
3. Interactive threshold slider
4. Threshold sweep table + clinical-target thresholds
5. Feature-detection diagnostics: kernels, receptive field, cross-lead attention, saliency intervals

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix,
)
from torch.utils.data import DataLoader, TensorDataset

from src.models.repnet_crosslead_deeper import RepNetCrossLeadDeeper
from src.data.dataset import load_seniordesign, split_holdout_grouped
from src.preprocessing.filters import BaselineWanderFilter, NotchFilter
from src.preprocessing.normalization import ZScoreNormalization

In [ ]:
MODEL_PATH = Path('../../optuna_crosslead_3stage_filters/2026-04-27_10-29-59/best_model.pt')
DATA_DIR   = '../../data/seniordesign_upload'
SEED       = 42

NET_PARAMS = dict(
    stage_filters = (48, 96, 192),    # filter study winner
    kernels       = (7, 5, 3),
    dropout       = 0.0546,
    n_heads       = 4,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Model:  {MODEL_PATH.resolve()}')

In [ ]:
net = RepNetCrossLeadDeeper(**NET_PARAMS).to(device)
net.load_state_dict(torch.load(MODEL_PATH, map_location=device))
net.eval()
n_params = sum(p.numel() for p in net.parameters())
print(f'Loaded. Parameters: {n_params:,}')

In [4]:
X, y, patient_ids = load_seniordesign(DATA_DIR, return_patient_ids=True)

flat_mask = (X.std(axis=2) < 1e-4).any(axis=1)
try:
    nan_mask = np.isnan(patient_ids.astype(float))
except (ValueError, TypeError):
    nan_mask = np.array([str(p).strip() in ('', 'nan', 'None') for p in patient_ids])
keep = ~flat_mask & ~nan_mask
X, y, patient_ids = X[keep], y[keep], patient_ids[keep]

X, _ = BaselineWanderFilter(cutoff=0.5, order=4, fs=250.0).transform(X)
X, _ = NotchFilter(freq=60.0, Q=30.0, fs=250.0).transform(X)
X, _ = ZScoreNormalization(per_lead=True).transform(X)

X_dev, X_test, y_dev, y_test, g_dev, g_test = split_holdout_grouped(
    X, y, patient_ids, test_size=0.20, seed=SEED,
)

print(f'Dev (training) : N={len(y_dev)}   PE={int(y_dev.sum())}   Normal={int((y_dev==0).sum())}   ({100*y_dev.mean():.1f}% pos)')
print(f'Test (holdout) : N={len(y_test)}   PE={int(y_test.sum())}   Normal={int((y_test==0).sum())}   ({100*y_test.mean():.1f}% pos)')

Dev (training) : N=1747   PE=277   Normal=1470   (15.9% pos)
Test (holdout) : N=431   PE=58   Normal=373   (13.5% pos)


In [5]:
def infer(net, X, device, batch_size=64):
    Xt = torch.tensor(X, dtype=torch.float32)
    dl = DataLoader(TensorDataset(Xt), batch_size=batch_size, num_workers=0)
    out = []
    with torch.no_grad():
        for (xb,) in dl:
            logits = net(xb.to(device))
            out.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

probs_dev  = infer(net, X_dev,  device)
probs_test = infer(net, X_test, device)

auroc_dev  = roc_auc_score(y_dev,  probs_dev)
auroc_test = roc_auc_score(y_test, probs_test)
auprc_dev  = average_precision_score(y_dev,  probs_dev)
auprc_test = average_precision_score(y_test, probs_test)

print(f'Train (dev) — AUROC: {auroc_dev:.4f}    AUPRC: {auprc_dev:.4f}')
print(f'Test (held) — AUROC: {auroc_test:.4f}    AUPRC: {auprc_test:.4f}')
print(f'Generalization gap: {auroc_dev - auroc_test:+.4f}')

Train (dev) — AUROC: 0.8296    AUPRC: 0.5516
Test (held) — AUROC: 0.7952    AUPRC: 0.3917
Generalization gap: +0.0344


## 1. Train vs test distribution (mirror histogram)

PE goes up, Normal goes down. Easy visual read of class separation.

In [6]:
def mirror_hist_traces(probs_, y_, nbins=40, show_legend=False):
    bins = np.linspace(0, 1, nbins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    width = bins[1] - bins[0]
    h_norm, _ = np.histogram(probs_[y_ == 0], bins=bins, density=True)
    h_pe,   _ = np.histogram(probs_[y_ == 1], bins=bins, density=True)
    return [
        go.Bar(x=centers, y=h_pe, name='PE',
               marker_color='tomato', width=width,
               showlegend=show_legend, legendgroup='PE'),
        go.Bar(x=centers, y=-h_norm, name='Normal',
               marker_color='steelblue', width=width,
               showlegend=show_legend, legendgroup='Normal'),
    ], h_norm.max(), h_pe.max()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=(f'Train  AUROC={auroc_dev:.3f}  N={len(y_dev)}',
                    f'Test   AUROC={auroc_test:.3f}  N={len(y_test)}'))
ymax = 0.0
for col, (probs_, y_) in enumerate([(probs_dev, y_dev), (probs_test, y_test)], start=1):
    traces, hn, hp = mirror_hist_traces(probs_, y_, show_legend=(col == 1))
    ymax = max(ymax, hn, hp)
    for tr in traces:
        fig.add_trace(tr, row=1, col=col)
    fig.add_vline(x=0.5, line=dict(dash='dash', color='black'), row=1, col=col)
ymax *= 1.1
tickvals = np.linspace(-ymax, ymax, 7)
ticktext = [f'{abs(v):.1f}' for v in tickvals]
fig.update_yaxes(tickvals=tickvals, ticktext=ticktext,
                 zeroline=True, zerolinecolor='black', zerolinewidth=1)
fig.update_layout(template='plotly_white', barmode='overlay', bargap=0,
                  title='P(PE) — train vs test (PE ↑, Normal ↓)',
                  width=1100, height=460)
fig.update_xaxes(title_text='P(PE)')
fig.update_yaxes(title_text='Density (PE up / Normal down)', col=1)
fig.show()

## 2. ROC + Precision-Recall (train vs test)

In [7]:
fpr_dv, tpr_dv, _ = roc_curve(y_dev,  probs_dev)
fpr_te, tpr_te, _ = roc_curve(y_test, probs_test)
prec_dv, rec_dv, _ = precision_recall_curve(y_dev,  probs_dev)
prec_te, rec_te, _ = precision_recall_curve(y_test, probs_test)

fig2 = make_subplots(rows=1, cols=2, subplot_titles=('ROC', 'Precision-Recall'))
fig2.add_trace(go.Scatter(x=fpr_dv, y=tpr_dv, name=f'Train (AUC={auroc_dev:.3f})',
                          line=dict(color='steelblue')), row=1, col=1)
fig2.add_trace(go.Scatter(x=fpr_te, y=tpr_te, name=f'Test (AUC={auroc_test:.3f})',
                          line=dict(color='tomato')), row=1, col=1)
fig2.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Random',
                          line=dict(dash='dash', color='gray'), showlegend=False), row=1, col=1)
fig2.add_trace(go.Scatter(x=rec_dv, y=prec_dv, name=f'Train (AP={auprc_dev:.3f})',
                          line=dict(color='steelblue', dash='dot')), row=1, col=2)
fig2.add_trace(go.Scatter(x=rec_te, y=prec_te, name=f'Test (AP={auprc_test:.3f})',
                          line=dict(color='tomato', dash='dot')), row=1, col=2)
fig2.update_xaxes(title_text='FPR', row=1, col=1)
fig2.update_yaxes(title_text='TPR', row=1, col=1)
fig2.update_xaxes(title_text='Recall', row=1, col=2)
fig2.update_yaxes(title_text='Precision', row=1, col=2)
fig2.update_layout(template='plotly_white', width=1100, height=480,
                   title='ROC & PR — train vs test')
fig2.show()

## 3. Interactive threshold slider (test set)

Drag τ to update the confusion matrix and metric panel.

In [8]:
 def metrics_at(probs_, y_, tau):
    pred = (probs_ >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_, pred, labels=[0, 1]).ravel()
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    ppv  = tp / max(tp + fp, 1)
    npv  = tn / max(tn + fn, 1)
    f1   = 2 * ppv * sens / max(ppv + sens, 1e-9)
    acc  = (tp + tn) / (tp + tn + fp + fn)
    return dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                sens=sens, spec=spec, ppv=ppv, npv=npv, f1=f1, acc=acc)

thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
metric_table = [metrics_at(probs_test, y_test, t) for t in thresholds]

p_norm = probs_test[y_test == 0]
p_pe   = probs_test[y_test == 1]

nbins = 40
bins = np.linspace(0, 1, nbins + 1)
centers = 0.5 * (bins[:-1] + bins[1:])
bar_w = bins[1] - bins[0]
h_norm, _ = np.histogram(p_norm, bins=bins, density=True)
h_pe,   _ = np.histogram(p_pe,   bins=bins, density=True)

fig3 = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38],
    specs=[[{'type': 'xy'}, {'type': 'heatmap'}]],
    subplot_titles=('Test P(PE) distribution  (PE ↑, Normal ↓)', 'Confusion matrix'))
fig3.add_trace(go.Bar(x=centers, y=h_pe, name='PE', marker_color='tomato', width=bar_w), row=1, col=1)
fig3.add_trace(go.Bar(x=centers, y=-h_norm, name='Normal', marker_color='steelblue', width=bar_w), row=1, col=1)

init_tau = 0.50
init_idx = int(np.argmin(np.abs(thresholds - init_tau)))
init_m = metric_table[init_idx]
cm_z = [[init_m['tn'], init_m['fp']], [init_m['fn'], init_m['tp']]]
fig3.add_trace(go.Heatmap(z=cm_z, x=['Pred Normal', 'Pred PE'], y=['True Normal', 'True PE'],
    colorscale='Blues', showscale=False, text=cm_z, texttemplate='%{text}', textfont={'size': 18}), row=1, col=2)
fig3.add_shape(type='line', x0=init_tau, x1=init_tau, y0=0, y1=1,
               yref='paper', xref='x', line=dict(color='black', dash='dash', width=2))

def metric_text(t, m):
    return (f'<b>τ = {t:.2f}</b><br>'
            f'Sensitivity : {m["sens"]:.3f}<br>'
            f'Specificity : {m["spec"]:.3f}<br>'
            f'PPV         : {m["ppv"]:.3f}<br>'
            f'NPV         : {m["npv"]:.3f}<br>'
            f'F1          : {m["f1"]:.3f}<br>'
            f'Accuracy    : {m["acc"]:.3f}')
fig3.add_annotation(text=metric_text(init_tau, init_m),
    xref='paper', yref='paper', x=0.40, y=0.98, align='left', showarrow=False,
    bgcolor='rgba(255,255,255,0.85)', bordercolor='gray', borderwidth=1, borderpad=8,
    font=dict(family='monospace', size=12))

steps = []
for i, t in enumerate(thresholds):
    m = metric_table[i]
    cm_new = [[m['tn'], m['fp']], [m['fn'], m['tp']]]
    steps.append(dict(method='update', label=f'{t:.2f}',
        args=[{'z': [None, None, [cm_new]], 'text': [None, None, [cm_new]]},
              {'shapes': [dict(type='line', x0=float(t), x1=float(t), y0=0, y1=1,
                               yref='paper', xref='x', line=dict(color='black', dash='dash', width=2))],
               'annotations[2].text': metric_text(t, m)}]))

ymax = max(h_pe.max(), h_norm.max()) * 1.1
tickvals = np.linspace(-ymax, ymax, 7)
ticktext = [f'{abs(v):.1f}' for v in tickvals]
fig3.update_layout(
    sliders=[dict(active=init_idx, currentvalue={'prefix': 'τ = '}, pad={'t': 50}, steps=steps)],
    barmode='overlay', bargap=0, template='plotly_white', width=1200, height=560,
    title=f'Threshold tuner — best CrossLead Deeper (test, AUROC={auroc_test:.3f})')
fig3.update_xaxes(title_text='P(PE)', row=1, col=1)
fig3.update_yaxes(title_text='Density (PE up / Normal down)', tickvals=tickvals, ticktext=ticktext,
                  zeroline=True, zerolinecolor='black', zerolinewidth=1, row=1, col=1)
fig3.show()

IndentationError: unexpected indent (2627034389.py, line 1)

## 4. Threshold sweep + clinical-target thresholds

In [ ]:
key_taus = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
rows = []
for t in key_taus:
    m = metrics_at(probs_test, y_test, t)
    rows.append({'τ': t, 'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'], 'TN': m['tn'],
                 'Sens': round(m['sens'], 3), 'Spec': round(m['spec'], 3),
                 'PPV': round(m['ppv'], 3), 'NPV': round(m['npv'], 3),
                 'F1': round(m['f1'], 3), 'Acc': round(m['acc'], 3)})
pd.DataFrame(rows).set_index('τ')

In [ ]:
def threshold_for_target(probs_, y_, target_sens=0.85):
    fpr, tpr, thr = roc_curve(y_, probs_)
    valid = tpr >= target_sens
    if not valid.any():
        return None
    i = int(np.argmax(thr[valid]))
    return float(thr[valid][i])

for target in [0.95, 0.90, 0.85, 0.80]:
    t = threshold_for_target(probs_test, y_test, target)
    if t is None:
        print(f'target sens {target}: not achievable')
        continue
    m = metrics_at(probs_test, y_test, t)
    print(f'sens >= {target} → τ={t:.3f}  sens={m["sens"]:.3f}  spec={m["spec"]:.3f}  '
          f'PPV={m["ppv"]:.3f}  F1={m["f1"]:.3f}  (TP={m["tp"]}, FP={m["fp"]}, FN={m["fn"]}, TN={m["tn"]})')

## Classification reports at key operating points

Sklearn's `classification_report` for the three operating points used above (default τ=0.5, Youden's J, and sens ≥ 0.85). Shows per-class precision, recall, F1, support, plus macro/weighted averages.

In [ ]:
from sklearn.metrics import classification_report

# Compute the three thresholds we care about
fpr_te, tpr_te, thr_te = roc_curve(y_test, probs_test)
tau_youden = float(thr_te[np.argmax(tpr_te - fpr_te)])

valid = tpr_te >= 0.85
tau_sens85 = float(thr_te[valid][np.argmax(thr_te[valid])]) if valid.any() else 0.0

operating_points = [
    ("tau = 0.50  (default)",          0.50),
    (f"tau = {tau_youden:.3f}  (Youden's J)", tau_youden),
    (f"tau = {tau_sens85:.3f}  (sens >= 0.85)", tau_sens85),
]

for name, tau in operating_points:
    pred = (probs_test >= tau).astype(int)
    print(f'\n=== {name} ===')
    print(classification_report(
        y_test, pred,
        target_names=['Normal', 'PE'],
        digits=3,
        zero_division=0,
    ))

---

# 5. Feature-detection diagnostics

## 5a. First-stage conv kernels

48 stage-1 kernels (k=7 = 28 ms @ 250 Hz). Look for derivative/peak/oscillatory shapes.

In [ ]:
k1 = net.stages[0]['conv'].conv1.weight.detach().cpu().squeeze(1).numpy()  # (48, 7)
order = np.argsort(-np.linalg.norm(k1, axis=1))
k1_sorted = k1[order]

fig_k = go.Figure(data=go.Heatmap(z=k1_sorted, colorscale='RdBu_r', zmid=0,
                                  colorbar=dict(title='weight')))
fig_k.update_layout(
    title=f'Stage 1 conv1 kernels ({k1.shape[0]} filters × {k1.shape[1]} taps), sorted by L2 norm',
    xaxis_title='kernel tap (4 ms each)', yaxis_title='filter idx (sorted)',
    template='plotly_white', width=620, height=620)
fig_k.show()

## 5b. Receptive field

In [ ]:
def rf_table(kernels, fs=250.0):
    rf, eff_stride = 1, 1
    rows = []
    for stage_idx, k in enumerate(kernels, start=1):
        rf += (k - 1) * eff_stride
        rows.append({'stage': stage_idx, 'layer': 'conv1', 'k': k, 'RF (samples)': rf,
                     'RF (ms)': round(1000*rf/fs, 1)})
        rf += (k - 1) * eff_stride
        rows.append({'stage': stage_idx, 'layer': 'conv2', 'k': k, 'RF (samples)': rf,
                     'RF (ms)': round(1000*rf/fs, 1)})
        eff_stride *= 2
        rows.append({'stage': stage_idx, 'layer': 'pool', 'k': '-', 'RF (samples)': rf,
                     'RF (ms)': round(1000*rf/fs, 1)})
    return pd.DataFrame(rows)

rf_df = rf_table(NET_PARAMS['kernels'], fs=250.0)
print(f"Total RF: {rf_df['RF (samples)'].iloc[-1]} samples = {rf_df['RF (ms)'].iloc[-1]} ms @ 250 Hz")
rf_df

## 5c. Cross-lead attention (per-stage, per-class)

In [ ]:
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
attn_storage = {f'stage{i+1}': [] for i in range(len(net.stages))}

def make_attn_hook(name):
    def hook(module, inputs, output):
        if isinstance(output, tuple) and len(output) >= 2 and output[1] is not None:
            attn_storage[name].append(output[1].detach().cpu().numpy())
    return hook

handles = []
for i, stage in enumerate(net.stages):
    handles.append(stage['attn'].attn.register_forward_hook(make_attn_hook(f'stage{i+1}')))

with torch.no_grad():
    Xt = torch.tensor(X_test, dtype=torch.float32)
    dl = DataLoader(TensorDataset(Xt), batch_size=64)
    for (xb,) in dl:
        net(xb.to(device))

for h in handles: h.remove()

attn_by_stage = {k: np.concatenate(v, axis=0) for k, v in attn_storage.items()}
n_stages = len(attn_by_stage)

fig_a = make_subplots(rows=2, cols=n_stages,
    subplot_titles=[f'Stage {i+1} — Normal' for i in range(n_stages)] +
                   [f'Stage {i+1} — PE' for i in range(n_stages)],
    horizontal_spacing=0.08, vertical_spacing=0.12)
for col, k in enumerate([f'stage{i+1}' for i in range(n_stages)], start=1):
    A = attn_by_stage[k]
    A_norm = A[y_test == 0].mean(axis=0)
    A_pe   = A[y_test == 1].mean(axis=0)
    fig_a.add_trace(go.Heatmap(z=A_norm, x=LEADS, y=LEADS,
                               colorscale='Blues', showscale=(col == n_stages),
                               colorbar=dict(x=1.02, len=0.4, y=0.78)),
                    row=1, col=col)
    fig_a.add_trace(go.Heatmap(z=A_pe, x=LEADS, y=LEADS,
                               colorscale='Reds', showscale=(col == n_stages),
                               colorbar=dict(x=1.02, len=0.4, y=0.22)),
                    row=2, col=col)
fig_a.update_layout(template='plotly_white', width=1200, height=700,
                    title='Mean cross-lead attention per stage (rows=query, cols=key)')
fig_a.show()

## 5d. Saliency heatmap (Integrated Gradients)Picks 8 examples covering {confident, borderline} × {TP, TN, FP, FN}, runs Integrated Gradients with σ=5 smoothing, and renders the signed attribution as a continuous diverging-colormap heatmap behind each lead's ECG trace.**Color reading:** red = pushes toward PE, blue = toward Normal. Color *intensity* = attribution magnitude (continuous, not thresholded).

In [ ]:
from scipy.ndimage import gaussian_filter1d

def integrated_gradients(net, x_np, target_class=1, n_steps=32, baseline=None, smooth_sigma=5.0):
    x = torch.tensor(x_np, dtype=torch.float32, device=device)
    base = (torch.zeros_like(x) if baseline is None
            else torch.tensor(baseline, dtype=torch.float32, device=device))
    alphas = torch.linspace(0.5/n_steps, 1.0-0.5/n_steps, n_steps, device=device).view(-1, 1, 1)
    interp = base.unsqueeze(0) + alphas * (x - base).unsqueeze(0)
    interp.requires_grad_(True)
    net.zero_grad()
    logits = net(interp)
    grads = torch.autograd.grad(logits[:, target_class].sum(), interp)[0]
    avg_grad = grads.mean(dim=0)
    attribution = ((x - base) * avg_grad).cpu().numpy()
    if smooth_sigma > 0:
        attribution = gaussian_filter1d(attribution, sigma=smooth_sigma, axis=1)
    return attribution

def extract_intervals(attr, threshold_frac=0.20, min_samples=5, fs=250.0):
    abs_attr = np.abs(attr)
    intervals = []
    for L in range(attr.shape[0]):
        lead_max = float(abs_attr[L].max())
        if lead_max == 0:
            continue
        thresh = lead_max * threshold_frac
        above = abs_attr[L] > thresh
        if not above.any():
            continue
        diff = np.diff(above.astype(int), prepend=0, append=0)
        starts = np.where(diff == 1)[0]
        ends   = np.where(diff == -1)[0]
        for s, e in zip(starts, ends):
            if (e - s) < min_samples:
                continue
            intervals.append({
                'lead': L, 't_start': s/fs, 't_end': e/fs,
                's_idx': int(s), 'e_idx': int(e),
                'sign': float(attr[L, s:e].mean()),
                'magnitude': float(abs_attr[L, s:e].mean()),
            })
    return intervals

def pick_8_cases(probs, y):
    pred = (probs >= 0.5).astype(int)
    tp = np.where((y == 1) & (pred == 1))[0]
    tn = np.where((y == 0) & (pred == 0))[0]
    fp = np.where((y == 0) & (pred == 1))[0]
    fn = np.where((y == 1) & (pred == 0))[0]
    out = {}
    if len(tp): out['confident_TP']  = tp[np.argmax(probs[tp])]
    if len(tn): out['confident_TN']  = tn[np.argmin(probs[tn])]
    if len(fp): out['confident_FP']  = fp[np.argmax(probs[fp])]
    if len(fn): out['confident_FN']  = fn[np.argmin(probs[fn])]
    if len(tp): out['borderline_TP'] = tp[np.argmin(np.abs(probs[tp] - 0.5))]
    if len(tn): out['borderline_TN'] = tn[np.argmin(np.abs(probs[tn] - 0.5))]
    if len(fp): out['borderline_FP'] = fp[np.argmin(np.abs(probs[fp] - 0.5))]
    if len(fn): out['borderline_FN'] = fn[np.argmin(np.abs(probs[fn] - 0.5))]
    return out

cases = pick_8_cases(probs_test, y_test)
print('Selected examples:')
for label, idx in cases.items():
    print(f'  {label:<18}  idx={idx:>4}  true={int(y_test[idx])}  P(PE)={probs_test[idx]:.3f}')

attributions = {idx: integrated_gradients(net, X_test[idx]) for idx in cases.values()}
intervals_by_idx = {idx: extract_intervals(attributions[idx]) for idx in cases.values()}
ordered = list(cases.items())

In [ ]:
def plot_heatmap_for_case(label, idx, top_k=4, fs=250.0):
    """Per-lead saliency heatmap: continuous diverging colormap behind the ECG line."""
    attr = attributions[idx]            # (12, T) signed IG attribution
    ecg  = X_test[idx]                  # (12, T)
    t    = np.arange(ecg.shape[1]) / fs

    # Rank leads by total |attribution|
    lead_strength = np.abs(attr).sum(axis=1)
    top_leads = list(np.argsort(-lead_strength)[:top_k])

    # Symmetric colormap range based on top leads only (ignore quiet leads)
    cmax = float(np.abs(attr[top_leads]).max()) or 1.0

    fig = make_subplots(rows=top_k, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=[f'Lead {LEADS[L]}' for L in top_leads])

    for r, L in enumerate(top_leads, start=1):
        ymin, ymax = float(ecg[L].min()), float(ecg[L].max())
        ypad = (ymax - ymin) * 0.15

        # Continuous heatmap (replaces the flat-rectangle bands).
        # z is (2, T) — two identical rows so the heatmap fills the band vertically.
        fig.add_trace(go.Heatmap(
            x=t, y=[ymin - ypad, ymax + ypad],
            z=[attr[L], attr[L]],
            colorscale='RdBu_r',  zmid=0, zmin=-cmax, zmax=cmax,
            showscale=(r == 1),
            colorbar=dict(title='IG attr<br>(signed)', x=1.02, len=0.9) if r == 1 else None,
            opacity=0.55,
            hovertemplate='t=%{x:.2f}s<br>attr=%{z:.3e}<extra></extra>',
        ), row=r, col=1)

        # ECG trace on top of the heatmap
        fig.add_trace(go.Scatter(
            x=t, y=ecg[L], mode='lines',
            line=dict(color='black', width=1.2), showlegend=False,
            hovertemplate='t=%{x:.2f}s<br>ECG=%{y:.3f}<extra></extra>',
        ), row=r, col=1)

    pred       = int(probs_test[idx] >= 0.5)
    true_label = 'PE' if y_test[idx] == 1 else 'Normal'
    pred_label = 'PE' if pred == 1 else 'Normal'
    correct    = pred == int(y_test[idx])
    fig.update_layout(
        template='plotly_white', width=1200, height=180 * top_k + 120,
        title=(f'<b>{label.replace("_", " ")}</b>  |  idx {idx}  |  '
               f'true={true_label}  pred={pred_label}  P(PE)={probs_test[idx]:.3f}  |  '
               f'{"correct" if correct else "WRONG"}<br>'
               f'<sub>Red = pushes toward PE, Blue = toward Normal, color intensity = attribution magnitude  |  '
               f'colormap range: ±{cmax:.2e}</sub>'),
    )
    fig.update_xaxes(title_text='time (s)', row=top_k, col=1)
    return fig


for label, idx in ordered:
    plot_heatmap_for_case(label, idx, top_k=4).show()

## 5e. Grad-CAM (alternative attribution)

Grad-CAM uses the gradient of P(PE) w.r.t. the **last convolutional layer's activations**, weighted by global mean. It's typically smoother and more "blob-like" than vanilla gradients or IG, which is what makes it look more like a classical heatmap. Resolution at the conv stage is coarser (T/8 = 312 timesteps) and is upsampled back to 2500 for display.

This section computes Grad-CAM at stage 3's conv output for the same 8 cases.

In [ ]:
import torch.nn.functional as F


def gradcam_per_lead(net, x_np, target_class=1, smooth_sigma=2.0):
    """Per-lead Grad-CAM at stage 3's PerLeadConvBlock output.

    Returns (12, T) signed attribution. Positive = pushes toward target_class.
    Smoothness comes from (a) coarse spatial resolution at conv stage, then
    (b) bilinear upsample to T, then (c) optional Gaussian smoothing.
    """
    x = torch.tensor(x_np, dtype=torch.float32, device=device).unsqueeze(0)
    x.requires_grad_(False)

    # Hook stage 3's conv output (post per-lead conv, pre attention)
    activations = {}
    def fwd_hook(_m, _i, o):
        activations['feat'] = o
        o.retain_grad()
    h = net.stages[-1]['conv'].register_forward_hook(fwd_hook)

    net.zero_grad()
    logits = net(x)
    target = logits[0, target_class]
    target.backward()
    h.remove()

    feat = activations['feat']                # (1, 12, C, T_out)
    grad = feat.grad                          # (1, 12, C, T_out)

    # Channel-wise mean of gradients = importance weights per channel
    weights = grad.mean(dim=(0, 3))           # (12, C)
    # Weighted sum across channels — keeps lead axis, collapses channel axis
    cam = (weights.unsqueeze(-1) * feat[0]).sum(dim=1)   # (12, T_out)

    # Upsample to original time resolution
    T = x_np.shape[1]
    cam_full = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(12, T),
                             mode='bilinear', align_corners=False)
    cam_full = cam_full.squeeze().detach().cpu().numpy()  # (12, T)

    if smooth_sigma > 0:
        cam_full = gaussian_filter1d(cam_full, sigma=smooth_sigma, axis=1)
    return cam_full


# Compute Grad-CAM for the same 8 cases
gradcams = {idx: gradcam_per_lead(net, X_test[idx], target_class=1) for idx in cases.values()}

print('Grad-CAM ranges per case:')
for label, idx in ordered:
    g = gradcams[idx]
    print(f'  {label:<18}  min={g.min():+.3e}  max={g.max():+.3e}  '
          f'|max|={np.abs(g).max():.3e}')

In [ ]:
def plot_gradcam_for_case(label, idx, top_k=4, fs=250.0):
    """Same heatmap-overlay style as IG, but using Grad-CAM attribution."""
    cam = gradcams[idx]                 # (12, T) signed
    ecg = X_test[idx]
    t   = np.arange(ecg.shape[1]) / fs

    lead_strength = np.abs(cam).sum(axis=1)
    top_leads = list(np.argsort(-lead_strength)[:top_k])
    cmax = float(np.abs(cam[top_leads]).max()) or 1.0

    fig = make_subplots(rows=top_k, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=[f'Lead {LEADS[L]}' for L in top_leads])

    for r, L in enumerate(top_leads, start=1):
        ymin, ymax = float(ecg[L].min()), float(ecg[L].max())
        ypad = (ymax - ymin) * 0.15
        fig.add_trace(go.Heatmap(
            x=t, y=[ymin - ypad, ymax + ypad],
            z=[cam[L], cam[L]],
            colorscale='RdBu_r', zmid=0, zmin=-cmax, zmax=cmax,
            showscale=(r == 1),
            colorbar=dict(title='Grad-CAM<br>(signed)', x=1.02, len=0.9) if r == 1 else None,
            opacity=0.55,
        ), row=r, col=1)
        fig.add_trace(go.Scatter(
            x=t, y=ecg[L], mode='lines',
            line=dict(color='black', width=1.2), showlegend=False,
        ), row=r, col=1)

    pred       = int(probs_test[idx] >= 0.5)
    true_label = 'PE' if y_test[idx] == 1 else 'Normal'
    pred_label = 'PE' if pred == 1 else 'Normal'
    correct    = pred == int(y_test[idx])
    fig.update_layout(
        template='plotly_white', width=1200, height=180 * top_k + 120,
        title=(f'<b>Grad-CAM — {label.replace("_", " ")}</b>  |  idx {idx}  |  '
               f'true={true_label}  pred={pred_label}  P(PE)={probs_test[idx]:.3f}  |  '
               f'{"correct" if correct else "WRONG"}<br>'
               f'<sub>Red = pushes toward PE, Blue = toward Normal  |  colormap: ±{cmax:.2e}</sub>'),
    )
    fig.update_xaxes(title_text='time (s)', row=top_k, col=1)
    return fig


for label, idx in ordered:
    plot_gradcam_for_case(label, idx, top_k=4).show()

## 5f. Layer-by-layer activation tracking (confident TP)

Picks the most-confident true positive from §5d and traces its representation through every layer of the network — input ECG → 3 stages of (conv → attention) → fusion → GAP → classifier.

**What you should see:**
- **Stage 1 conv (48 channels):** filter responses to local QRS/T-wave shapes (RF=52 ms after the block).
- **Stage 1 attn:** same shape but element-wise gated by the cross-lead attention's sigmoid output. Difference = what attention does at this depth.
- **Stages 2 and 3:** progressively coarser temporal resolution (1250 → 625 → 312 samples) and more abstract feature channels (96 → 192).
- **Fusion (192 channels post-concat):** lead identity is gone, only abstract features remain.
- **GAP (192-dim vector):** single vector summarizing the entire 10-second ECG.
- **Linear (2 logits):** final class scores; softmax → P(PE).

In [ ]:
# Capture activations at every stage via forward hooks for the confident-TP sample.

TRACK_LEAD = 1   # 0=I, 1=II, 2=III, 3=aVR, 4=aVL, 5=aVF, 6=V1, 7=V2, 8=V3, 9=V4, 10=V5, 11=V6
TRACK_IDX  = cases['confident_TP']
print(f'Tracking sample idx={TRACK_IDX}, lead={LEADS[TRACK_LEAD]}, P(PE)={probs_test[TRACK_IDX]:.3f}')

acts = {}

def cap(name):
    def hook(_m, _i, output):
        if isinstance(output, tuple):
            output = output[0]
        acts[name] = output.detach().cpu()
    return hook

handles = [
    net.stages[0]['conv'].register_forward_hook(cap('s1_conv')),
    net.stages[0]['attn'].register_forward_hook(cap('s1_attn')),
    net.stages[1]['conv'].register_forward_hook(cap('s2_conv')),
    net.stages[1]['attn'].register_forward_hook(cap('s2_attn')),
    net.stages[2]['conv'].register_forward_hook(cap('s3_conv')),
    net.stages[2]['attn'].register_forward_hook(cap('s3_attn')),
    net.fuse.register_forward_hook(cap('fuse')),
    net.gap.register_forward_hook(cap('gap')),
    net.fc.register_forward_hook(cap('fc')),
]

with torch.no_grad():
    x = torch.tensor(X_test[TRACK_IDX:TRACK_IDX+1], dtype=torch.float32, device=device)
    logits = net(x)
    probs = torch.softmax(logits, dim=1)

for h in handles:
    h.remove()

print(f'\nLogits = {logits.cpu().numpy().ravel()}')
print(f'Softmax = {probs.cpu().numpy().ravel()}  -> P(PE) = {probs[0, 1].item():.4f}')
print(f'\nActivation tensor shapes (after each layer):')
for name, t in acts.items():
    print(f'  {name:<10}  {tuple(t.shape)}')

In [ ]:
# Per-stage activation heatmaps for the chosen lead.

ecg_lead = X_test[TRACK_IDX, TRACK_LEAD, :]                       # (2500,)
fs       = 250.0
t_input  = np.arange(2500) / fs

stages = [
    ('Stage 1: PerLeadConv (48 ch)',     's1_conv'),
    ('Stage 1: CrossLeadAttn output',    's1_attn'),
    ('Stage 2: PerLeadConv (96 ch)',     's2_conv'),
    ('Stage 2: CrossLeadAttn output',    's2_attn'),
    ('Stage 3: PerLeadConv (192 ch)',    's3_conv'),
    ('Stage 3: CrossLeadAttn output',    's3_attn'),
]

n_rows = 1 + len(stages)
fig = make_subplots(
    rows=n_rows, cols=1, shared_xaxes=False,
    vertical_spacing=0.025,
    row_heights=[0.10] + [0.15] * len(stages),
    subplot_titles=[f'Input ECG  Lead {LEADS[TRACK_LEAD]}  (2500 samples = 10 s)'] +
                   [f'{name}  --  Lead {LEADS[TRACK_LEAD]}  --  shape {tuple(acts[k].shape[2:])}'
                    for name, k in stages],
)

# Row 1: input ECG line
fig.add_trace(go.Scatter(x=t_input, y=ecg_lead, mode='lines',
                         line=dict(color='black', width=1.2), showlegend=False,
                         name='ECG'), row=1, col=1)

# Rows 2..N: per-stage activations as channel x time heatmap
for r, (name, key) in enumerate(stages, start=2):
    a = acts[key][0, TRACK_LEAD].numpy()   # (C, T_stage)
    C, T_stage = a.shape
    t_stage = np.linspace(0, 10.0, T_stage)
    cmax = float(np.abs(a).max()) or 1.0
    fig.add_trace(go.Heatmap(
        x=t_stage, y=np.arange(C), z=a,
        colorscale='RdBu_r', zmid=0, zmin=-cmax, zmax=cmax,
        showscale=False,
        hovertemplate='ch=%{y}<br>t=%{x:.2f}s<br>act=%{z:.3f}<extra></extra>',
    ), row=r, col=1)

fig.update_xaxes(range=[0, 10.0])
fig.update_yaxes(title_text='mV (z-scored)', row=1, col=1)
for r in range(2, n_rows + 1):
    fig.update_yaxes(title_text='channel', row=r, col=1)
fig.update_xaxes(title_text='time (s)', row=n_rows, col=1)

fig.update_layout(
    template='plotly_white',
    width=1200, height=180 * n_rows + 100,
    title=(f'<b>Layer-by-layer activations</b>  |  '
           f'idx {TRACK_IDX} (confident TP)  |  '
           f'lead {LEADS[TRACK_LEAD]}  |  '
           f'P(PE) = {probs_test[TRACK_IDX]:.3f}<br>'
           f'<sub>Each row = activation map after that layer (channel index on y, time on x). '
           f'Diverging colormap (red = positive, blue = negative).</sub>'),
)
fig.show()


# Fusion + classifier-head outputs (no per-lead axis anymore)
fuse_out = acts['fuse'][0].numpy()          # (192, T_stage)
gap_out  = acts['gap'][0, :, 0].numpy()     # (192,)
fc_out   = acts['fc'][0].numpy()            # (2,)
softmax_out = probs[0].cpu().numpy()        # (2,)

C_fuse, T_fuse = fuse_out.shape
t_fuse = np.linspace(0, 10.0, T_fuse)
cmax_fuse = float(np.abs(fuse_out).max()) or 1.0

fig2 = make_subplots(
    rows=3, cols=1,
    row_heights=[0.55, 0.25, 0.20],
    vertical_spacing=0.10,
    subplot_titles=(
        f'Fusion output  ({C_fuse} channels x {T_fuse} time)  --  per-lead identity gone',
        f'GAP output  ({len(gap_out)}-dim vector  --  one number per channel)',
        f'Logits  -->  Softmax  -->  P(PE) = {softmax_out[1]:.4f}',
    ),
)

fig2.add_trace(go.Heatmap(
    x=t_fuse, y=np.arange(C_fuse), z=fuse_out,
    colorscale='RdBu_r', zmid=0, zmin=-cmax_fuse, zmax=cmax_fuse,
    showscale=True, colorbar=dict(title='fusion act', x=1.02, len=0.5, y=0.78),
), row=1, col=1)

gap_colors = ['tomato' if v > 0 else 'steelblue' for v in gap_out]
fig2.add_trace(go.Bar(
    x=np.arange(len(gap_out)), y=gap_out,
    marker_color=gap_colors, showlegend=False,
), row=2, col=1)
fig2.add_hline(y=0, line=dict(color='black', width=1), row=2, col=1)

fig2.add_trace(go.Bar(
    x=['Normal logit', 'PE logit', 'Normal P', 'PE P'],
    y=[fc_out[0], fc_out[1], softmax_out[0], softmax_out[1]],
    marker_color=['steelblue', 'tomato', 'lightblue', 'lightcoral'],
    text=[f'{fc_out[0]:.2f}', f'{fc_out[1]:.2f}',
          f'{softmax_out[0]:.3f}', f'{softmax_out[1]:.3f}'],
    textposition='outside',
    showlegend=False,
), row=3, col=1)

fig2.update_xaxes(title_text='time (s)', row=1, col=1)
fig2.update_yaxes(title_text='channel', row=1, col=1)
fig2.update_xaxes(title_text='channel', row=2, col=1)
fig2.update_yaxes(title_text='avg activation', row=2, col=1)

fig2.update_layout(
    template='plotly_white', width=1200, height=900,
    title=f'<b>Fusion + classifier head</b>  |  idx {TRACK_IDX} (confident TP)  |  P(PE) = {probs_test[TRACK_IDX]:.4f}',
)
fig2.show()

## Reading the diagnostics

- **Kernels**: structured shapes (peaks, derivatives, oscillations) → real low-level features.
- **Receptive field**: 180 ms — covers QRS plus early ST segment.
- **Cross-lead attention**: if Normal vs PE matrices look identical, attention isn't class-discriminative.
  Stage 3 attention is most class-relevant since it operates on the highest-level features.
- **Saliency intervals**: should concentrate on QRS / T-wave regions in PE cases.
  Compare confident_TP and borderline_TP — same intervals or different?
  Compare confident_TP and confident_FP — model gets fooled by which features?